In [ ]:
import os
from pathlib import Path
from collections import Counter

# ⚠️ 注意：这里填入你 datasets 文件夹的绝对路径
TARGET_BASE_DIR = Path(r"D:\MyProject\smart-campus-lnf\yolo_model\datasets")

def check_dataset(split_name="val"):
    print(f"\n" + "="*15 + f" 正在体检: {split_name.upper()} 集 " + "="*15)
    
    img_dir = TARGET_BASE_DIR / 'images' / split_name
    lbl_dir = TARGET_BASE_DIR / 'labels' / split_name

    if not img_dir.exists() or not lbl_dir.exists():
        print(f"❌ 找不到 {split_name} 的图片或标签文件夹！请检查路径。")
        return

    # 1. 搜集所有文件名（拔掉后缀）
    images = {f.stem for f in img_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.jpeg', '.png']}
    labels = {f.stem for f in lbl_dir.glob('*.txt')}

    # 2. 检查匹配情况 (集合运算的魔法)
    img_without_lbl = images - labels
    lbl_without_img = labels - images

    print(f"📸 共有图片: {len(images)} 张")
    print(f"📝 共有标签: {len(labels)} 个")

    if img_without_lbl:
        print(f"⚠️ 注意: 有 {len(img_without_lbl)} 张图片没有标签文件。")
        print("   (如果这些全是你放进去的纯背景图，那就是正常的✅)")
    else:
        print("✅ 所有图片都有对应的标签文件。")

    if lbl_without_img:
        print(f"🚨 致命错误: 有 {len(lbl_without_img)} 个标签找不到对应的图片！")
    else:
        print("✅ 没有“幽灵”标签。")

    # 3. 暴力解剖：统计所有 txt 里的类别编号
    class_counts = Counter()
    for lbl_name in labels:
        lbl_path = lbl_dir / (lbl_name + '.txt')
        # 忽略大小为0的空文件（背景图的空标签）
        if os.path.getsize(lbl_path) == 0:
            continue
            
        with open(lbl_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if parts: # 如果这一行有数据
                    class_id = int(parts[0])
                    class_counts[class_id] += 1

    # 4. 生成体检报告 (我们期望有 0 到 9 这 10 个类别)
    print("\n📈 类别分布统计 (检测框总数):")
    missing_classes = []
    
    for i in range(10): # 0 到 9
        count = class_counts.get(i, 0)
        if count == 0:
            missing_classes.append(i)
            print(f"  ❌ 类别 {i}: 0 个 (完全缺失！)")
        else:
            print(f"  ✅ 类别 {i}: {count} 个")

    if missing_classes:
        print(f"\n🚨 结论: {split_name.upper()} 集中缺失了类别 {missing_classes}！")
    else:
        print(f"\n🎉 结论: {split_name.upper()} 集包含了所有 10 个类别，非常健康！")

if __name__ == '__main__':
    # 先查训练集，再查验证集
    check_dataset('train')
    check_dataset('val')


=============== 正在体检: TRAIN 集 ===============
📸 共有图片: 4506 张
📝 共有标签: 4506 个
✅ 所有图片都有对应的标签文件。
✅ 没有“幽灵”标签。

📈 类别分布统计 (检测框总数):
  ✅ 类别 0: 415 个
  ✅ 类别 1: 148 个
  ✅ 类别 2: 85 个
  ✅ 类别 3: 548 个
  ✅ 类别 4: 458 个
  ✅ 类别 5: 92 个
  ✅ 类别 6: 1486 个
  ✅ 类别 7: 218 个
  ✅ 类别 8: 1202 个
  ✅ 类别 9: 1769 个

🎉 结论: TRAIN 集包含了所有 10 个类别，非常健康！

=============== 正在体检: VAL 集 ===============
📸 共有图片: 1124 张
📝 共有标签: 1124 个
✅ 所有图片都有对应的标签文件。
✅ 没有“幽灵”标签。

📈 类别分布统计 (检测框总数):
  ✅ 类别 0: 102 个
  ✅ 类别 1: 36 个
  ✅ 类别 2: 20 个
  ✅ 类别 3: 127 个
  ✅ 类别 4: 113 个
  ✅ 类别 5: 35 个
  ✅ 类别 6: 342 个
  ✅ 类别 7: 68 个
  ✅ 类别 8: 290 个
  ✅ 类别 9: 449 个

🎉 结论: VAL 集包含了所有 10 个类别，非常健康！


: 